# PDB 6EQE — IsPETase structural basics

First contact with the IsPETase crystal structure (Austin et al. 2018).

Goal: load the PDB file, understand the structure hierarchy, find the
catalytic triad residues (Ser-160, Asp-206, His-237), and verify they
form a real active site by checking that their atoms are physically close
in 3D space.

In [1]:
from Bio.PDB import PDBParser
import numpy as np

parser = PDBParser(QUIET=True)
structure = parser.get_structure("6EQE", "../data/6eqe.pdb")
print(f"Loaded structure: {structure.id}")

Loaded structure: 6EQE


Structure
└── Model (usually just 1)
    └── Chain (e.g., chain A, chain B...)
        └── Residue (each amino acid)
            └── Atom (each individual atom)

In [2]:
# How many models does the structure have?
models = list(structure.get_models())
print(f"Number of models: {len(models)}")

# How many chains in the first model?
chains = list(structure[0].get_chains())
print(f"Chains in model 0: {[chain.id for chain in chains]}")

# How many residues in chain A?
chain_a = structure[0]["A"]
residues = list(chain_a.get_residues())
print(f"Residues in chain A: {len(residues)}")

Number of models: 1
Chains in model 0: ['A']
Residues in chain A: 662


In [3]:
chain_a = structure[0]["A"]

# Get residue 160 (Ser in IsPETase)
res_160 = chain_a[160]
print(f"Residue at position 160: {res_160}")
print(f"Residue name: {res_160.get_resname()}")
print(f"Residue ID: {res_160.id}")

Residue at position 160: <Residue SER het=  resseq=160 icode= >
Residue name: SER
Residue ID: (' ', 160, ' ')


In [4]:
print(f"Atoms in residue 160 (Ser):")
for atom in res_160:
    print(f"  {atom.get_name()}: {atom.coord}")

Atoms in residue 160 (Ser):
  N: [-17.613  -7.764   7.103]
  CA: [-18.287  -9.034   7.361]
  C: [-17.323 -10.186   7.068]
  O: [-16.259 -10.209   7.691]
  CB: [-19.663  -9.07    6.709]
  OG: [-20.465 -10.036   7.369]
  H: [-17.499  -7.284   7.808]
  HA: [-18.453  -9.073   8.316]
  HB2: [-20.08   -8.198   6.789]
  HB3: [-19.57   -9.315   5.775]
  HG: [-20.108 -10.794   7.309]


In [5]:
for resnum, expected in [(206, "ASP"), (237, "HIS")]:
    res = chain_a[resnum]
    actual = res.get_resname()
    match = "✓" if actual == expected else "✗"
    print(f"{match} Residue {resnum}: {actual} (expected {expected})")

✓ Residue 206: ASP (expected ASP)
✓ Residue 237: HIS (expected HIS)


In [7]:
def distance(atom1, atom2):
    """Euclidean distance between two atoms in Ångstroms."""
    return np.linalg.norm(atom1.coord - atom2.coord)

# Catalytic triad distances
d_ser_his = distance(ser_og, his_ne2)
d_his_asp = distance(his_ne2, asp_od2)
d_ser_asp = distance(ser_og, asp_od2)

print(f"Ser-160 OG ←→ His-237 NE2: {d_ser_his:.2f} Å")
print(f"His-237 NE2 ←→ Asp-206 OD2: {d_his_asp:.2f} Å")
print(f"Ser-160 OG ←→ Asp-206 OD2:  {d_ser_asp:.2f} Å")

Ser-160 OG ←→ His-237 NE2: 2.94 Å
His-237 NE2 ←→ Asp-206 OD2: 4.77 Å
Ser-160 OG ←→ Asp-206 OD2:  6.21 Å


In [6]:
ser_og = chain_a[160]["OG"]
asp_od2 = chain_a[206]["OD2"]
his_ne2 = chain_a[237]["NE2"]

print(f"Ser-160 OG coord: {ser_og.coord}")
print(f"Asp-206 OD2 coord: {asp_od2.coord}")
print(f"His-237 NE2 coord: {his_ne2.coord}")

Ser-160 OG coord: [-20.465 -10.036   7.369]
Asp-206 OD2 coord: [-23.162  -6.76   11.9  ]
His-237 NE2 coord: [-22.843  -8.3     7.396]


In [8]:
# Pick a residue far from the active site, e.g., residue 50
ser_og = chain_a[160]["OG"]
other_ca = chain_a[50]["CA"]

d_to_50 = distance(ser_og, other_ca)
print(f"Ser-160 OG ←→ Residue 50 CA: {d_to_50:.2f} Å")

Ser-160 OG ←→ Residue 50 CA: 31.09 Å


In [9]:
# His-237 has two ring nitrogens: NE2 and ND1
# ND1 typically points toward Asp in serine hydrolase catalytic triads

his_nd1 = chain_a[237]["ND1"]

d_his_nd1_asp = distance(his_nd1, asp_od2)
d_his_nd1_od1 = distance(his_nd1, chain_a[206]["OD1"])

print(f"His-237 ND1 ←→ Asp-206 OD2: {d_his_nd1_asp:.2f} Å")
print(f"His-237 ND1 ←→ Asp-206 OD1: {d_his_nd1_od1:.2f} Å")

His-237 ND1 ←→ Asp-206 OD2: 2.66 Å
His-237 ND1 ←→ Asp-206 OD1: 3.07 Å


## Catalytic triad verification — summary

Verified IsPETase catalytic triad geometry from PDB 6EQE:

| Interaction | Distance | Interpretation |
|---|---|---|
| Ser-160 OG → His-237 NE2 | 2.94 Å | Hydrogen bond (Ser-His relay) |
| His-237 ND1 → Asp-206 OD1/OD2 | ~3 Å (re-measure) | Hydrogen bond (His-Asp relay) |
| Ser-160 OG → Asp-206 | 6.21 Å | No direct contact (His is bridge) |

This is the textbook serine hydrolase catalytic triad geometry. These
three residues do the chemistry of PET ester bond cleavage. They are
the residues NOT to mutate in this project — touching any of them
would kill enzyme activity.

Mutations should target stability residues (surface, loops, hydrophobic
core) while preserving this active site.

In [12]:
import py3Dmol

# Load the structure into a py3Dmol viewer
with open("../data/6eqe.pdb") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=500, height=500)
view.addModel(pdb_data, "pdb")

# Show the whole protein as a faint cartoon
view.setStyle({}, {"cartoon": {"color": "lightgray", "opacity": 0.6}})

# Highlight the catalytic triad with sticks
view.setStyle({"resi": [160, 206, 237]},
              {"stick": {"colorscheme": "yellowCarbon"},
               "cartoon": {"color": "yellow"}})

# Add residue labels
for resnum, name in [(160, "Ser160"), (206, "Asp206"), (237, "His237")]:
    view.addLabel(name,
                  {"backgroundColor": "black", "fontColor": "white", "fontSize": 12},
                  {"resi": resnum})

view.zoomTo({"resi": [160, 206, 237]})
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.